<a href="https://colab.research.google.com/github/tejalkv0/RAG-Bankingbot/blob/main/RAG_Bankbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install langchain_huggingface

In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 8.6 MB/s eta 0:00:00


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.7 MB/s eta 0:00:00


In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs = [
    "The State Bank of India (SBI) is India’s largest public sector bank and financial services organization with a legacy of over 200 years that offers a wide range of financial products and services to customers, including loans, accounts, cards, investments, deposits, and net banking. "
]

# Crucial Step: Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = ["https://bankingvista.com/apply-personal-loan/","https://bankingvista.com/lifetime-free-credit-cards/","https://bankingvista.com/calculators/","https://sbi.bank.in/web/about-us","https://sbi.bank.in/web/personal-banking/cards/debit-card"]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "sbi.pdf" # Instruct students to upload a PDF with this name
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("Genai")
)

print("\nɕ Unified Universal Bot Online. Ask about the bank, privacy , or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context.
    If the context does not contain the answer, say "I do not have enough information."
    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

Step 1: Ingesting data from multiple sources...
PDF loaded successfully.
Total raw pages/documents loaded: 12
Data chopped into 270 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


ɕ Unified Universal Bot Online. Ask about the bank, privacy , or PDF.
------------------------------------------------------------
AI: I do not have enough information.
AI: I do not have enough information.
